# Trees and Regularization

This notebook opens Chapter 8 and connects it to Chapter 6. We meet **decision trees**, which predict by evaluating a series of yes-or-no questions about the features.<sup>1</sup> A tree that is too deep can overfit the training data and score badly on new data, just like KNN with $K = 1$. Any technique used to mitigate that problem is called **regularization**. Trees allow both possible kinds of regularization: limits and penalty.

### By the end, you'll be able to

1. Describe how a tree predicts: follow the splits to a leaf, then predict the leaf's average (regression) or majority class (classification). Read a tree diagram and the matching picture of boxes.
2. Describe recursive binary splitting: try every feature and split point, keep the split that lowers training error the most, then repeat. Explain why this is a **greedy** approach.
3. Evaluate one candidate split by hand on a small data set.
4. Explain why a tree grown until every leaf is *pure* has 0% training error, and why its test error is poor, connecting it to $K = 1$.
5. State the broad definition of **regularization**, and classify a technique as a **limit** or a **penalty**.
6. Name scikit-learn's limit settings for trees and say which modification make a tree more or less flexible.
7. Explain cost-complexity pruning in words and as a formula, predict what happens as $\alpha$ grows, and choose $\alpha$ by cross-validation.
8. Distinguish the broad meaning of regularization from the narrower one in ISL Chapter 6.

---

<sup>1</sup> Some tree algorithms, such as C4.5 and CHAID, split a node into more than two branches. In this class we only use binary trees, where every split is a yes-or-no question. These are the trees in ISL and scikit-learn.


> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as pandas (`pd.`) and scikit-learn, do everyday tasks you'll do yourself: fitting a tree, computing an error rate, building a small table. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_6_and_8_demos_helpers` draw a plot or build an interactive demo. That file was written for this course, and its plotting code isn't the point. You never need to open it.

**Why study trees?** 

Trees are old: botanists have used hand-built yes/no keys since the 1700s, and the first program to grow a tree from data appeared in the early 1960s. A single tree is easy to read, since its prediction is a short list of if-then rules. Single trees like this often perform poorly, but they're great tools for introducing various concepts, or extending to more powerful models based on them. In *Ridge, Lasso, and Ensembles* we combine hundreds of trees into ensembles (random forests and boosting). Trees can also aid in feature-selection. Another crucial topic we'll learn from trees is **regularization**, which is a fundamental concept we'll use to mitigate the tendency of neural networks to memorize training data.

In [ ]:
import numpy as np
import pandas as pd

## 1 · How a tree predicts

Before we explore how to grow a tree from data (that is, write code that uses a greedy approach to constructing a tree), we'll focus on how an existing decision tree makes predictions.

A tree predicts by asking yes-or-no questions about the features, one at a time, such as "Is `Years` less than 4.5?". Each answer constrains the prediction to one sub-tree, where the next question is evaluated, until the model reaches a **leaf** (also called a **terminal node**)

- In regression, the prediction is the average response of the training points in that leaf.
- In classification, the prediction is the most common class in that leaf.
- Each question is called a **split**, and the questions that aren't leaves are **internal nodes**.

Every leaf covers a box (or hyper-box) in [feature space](../../Reference_Material/vocabulary.md#feature-space), so a tree cuts the feature space into boxes $R_1, \dots, R_J$, one per leaf. ISL calls these boxes **regions**.

### Prediction Task: Baseball Salaries

We'll start with ISL's own example (Figures 8.1 and 8.2): predicting a baseball player's salary from `Years` (seasons in the major leagues) and `Hits` (hits last season).

**Code worth reading.** `load_data('Hitters')` loads ISL's baseball data. 59 players have no salary listed. <details>

 </details> The model can't learn from a player whose salary we don't know, so we drop those rows. 

**Note:** There are many ways to deal with missing *features*, but if the $Y-value$ for a row is missing, the model can't learn any relationship between its features and ground truth ($Y-value$). That's why we drop rows for players with a missing salary.

 Salaries are in thousands of dollars. We'll predict the **log** of salary, which reduces the importance of players at the upper extreme of the salary range. 

<details>
<summary>Optional: Why log salary? Modeling choice details...</summary>

**Trees ignore the scale of the features, but not of the response.** A split like "`Years` $< 4.5$" depends only on the order of the `Years` values, so standardizing or taking the log of a feature wouldn't change the tree. The response is different: each split is chosen to lower the RSS, $\sum_i (y_i - \hat{y})^2$, and each leaf predicts the average of its $y_i$. Both depend on the actual sizes of the $y_i$, so changing $Y$ changes the tree.

**On the raw scale, a few players control the RSS.** Salaries run from \$68K to \$2.46M. Squaring the errors magnifies the gap: the 10 highest-paid of these 263 players account for 42% of the total sum of squares. A tree fit to raw salary gets most of its RSS reduction by sorting out those few players. In a two-level tree on raw salary, one of the three splits is spent isolating a single player.

**On the log scale, errors are relative.** The difference between two logs depends only on their ratio:

$$\log(110) - \log(100) = \log(1{,}100) - \log(1{,}000) = \log(1.1) \approx 0.095.$$

Predicting \$110K for a \$100K player and predicting \$1.1M for a \$1M player are both 10% off, so they cost the same. On the raw scale the second miss costs $100$ times as much ($100^2$ against $10^2$ in squared error).

**Why not standardize salary instead?** Standardizing replaces each $y_i$ with $(y_i - \bar{y})/s$. That is a shift and a rescale, so every error is divided by the same number $s$ and every squared error by $s^2$. The RSS of every candidate split shrinks by the same factor, so RSS is still dominated by the upper extreme of the pay scale. 

**Which is right depends on what you care about.** Taking the log is a decision about which mistakes matter:

- If a 10% miss is equally bad for every player (say, you want a fair estimate for each player's contract talks), predict log salary.
- If dollars are what matter (say, a team budgeting total payroll, where missing a star's salary by \$500K really is worse than missing a rookie's by \$10K), predict raw salary.

Neither is "correct". The data doesn't decide this; the person designing the model does, based on what the predictions are for. ISL chooses the log, and we'll do that for consistency. Just know that this model is better for the contract negotiation task and worse for team budgeting.

**One side effect.** A leaf predicts the average of the *log* salaries. Converting back with $e^{\hat{y}}$ gives a value below the leaf's average salary, closer to a typical player's salary in that leaf than to the mean. Keep that in mind when we read a leaf's value as dollars below.

</details>

In [ ]:
from ISLP import load_data

hitters = load_data('Hitters')
hitters = hitters.dropna(subset=['Salary'])     # drop the 59 players with no salary
X_hit = hitters[['Years', 'Hits']]
y_hit = np.log(hitters['Salary'])               # log of salary (in thousands of dollars)
print(f'{len(hitters)} players')

<a id="demo-a"></a>

### Demo A: How a tree predicts

**Code worth reading.** `DecisionTreeRegressor` fits a regression tree. `max_leaf_nodes=3` stops it at three leaves so the picture stays small (Section 5 says more about settings like this). `random_state=0` makes the result repeatable.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

small_tree = DecisionTreeRegressor(max_leaf_nodes=3, random_state=0).fit(X_hit, y_hit)

The left panel is the tree. Each box shows its question (if it has one), the number of training players that reach it (`samples`), and their mean log salary (`value`). The right panel shows the same tree as boxes in the `Years`–`Hits` plane: each leaf on the left has a box of the same color on the right.

The tree starts as a single leaf that predicts the same salary for everyone. Click **Next split ▶** to add one split at a time. The newest split is outlined in orange in the tree and drawn as a thick orange line among the boxes. Each split cuts one box into two, so the plane always has one box per leaf.

Behind the button, the demo refits the tree with `max_leaf_nodes` one larger each time. With that setting, scikit-learn adds splits in order of usefulness: of all the current leaves, it splits the one whose best split lowers the training RSS the most. (Section 2 shows how that best split is found.)

Click until the tree has **three leaves** (two splits): that's `small_tree`, and the question below uses it.

In [ ]:
from ch_6_and_8_demos_helpers import grow_tree_demo

grow_tree_demo(X_hit, y_hit, max_leaves=6)

**Ask:** A player has 6 years in the league and 150 hits last season. Which box is he in, and what salary does the tree predict?

<details>
<summary>Answer</summary>

Start at the top. Is `Years` $\le 4.5$? No (6), so go right. Is `Hits` $\le 117.5$? No (150), so go right again. That's the leaf with value 6.74, the green box in the upper right. The tree predicts a log salary of 6.74, which is about \$845,000 ($e^{6.74} \approx 845$, in thousands). Every player in that box gets the same prediction.

</details>

**Code worth reading.** `predict` does the same walk down the tree. It wants the same columns the tree was fit on, so we pass a one-row DataFrame.

In [ ]:
new_player = pd.DataFrame({'Years': [6], 'Hits': [150]})
log_salary = small_tree.predict(new_player)[0]
print(f'Predicted log salary: {log_salary:.2f}, about ${np.exp(log_salary):,.0f}K')

<a id="section-2"></a>

## 2 · How a tree is grown: the best split, greedily

To make the first split, the computer tries every feature $X_j$ and every **split point** $s$. A split on feature $j$ at split point $s$ cuts the feature space into two boxes (ISL equation 8.2):

$$
R_1(j, s) = \{X \mid X_j < s\} \quad\text{and}\quad R_2(j, s) = \{X \mid X_j \ge s\}
$$

Read $R_1(j, s)$ as "the set of all points $X$ such that $X_j < s$." The curly braces mean a **set**, and the vertical bar inside them means **"such that"**. This is *not* the "given" bar from conditional probability, as in $P(Y = 1 \mid X = x)$; that bar never appears inside curly braces. (See [Sets](../../Reference_Material/notation.md#sets) in the notation reference.)

Each box predicts its own mean, $\hat y_{R_1}$ or $\hat y_{R_2}$. The score of a split is the total RSS of the two boxes (ISL equation 8.3):

$$
\text{RSS} = \sum_{i:\, x_i \in R_1(j, s)} (y_i - \hat y_{R_1})^2 + \sum_{i:\, x_i \in R_2(j, s)} (y_i - \hat y_{R_2})^2
$$

Read the first sum as "the sum over every $i$ such that $x_i$ is in $R_1(j, s)$": one term for each training point in the left box. The colon is another way to write "such that." (See [A condition under a sum](../../Reference_Material/notation.md#sum-over-a-set).)

In each region, $R$:
1. add up the squared distances from each point's response ($y_i = \log(salary_i)$) to the box's mean log salary $\hat y_{R_m}$.
2. add the two boxes' totals. 
3. The split with the lowest RSS wins. 

This is the same RSS you've used since Chapter 3, with "the box's mean" as the prediction.

### Worked example: one split by hand

Before the real data, let's find a best split on a data set small enough to check by hand: five made-up players, with one feature (`Years`) and the response (`Salary`). To keep the arithmetic easy, salary is in hundreds of thousands of dollars (so 3 means \$300K), and we skip the log.

**Code worth reading.** A DataFrame with one column per variable and one row per player.

In [ ]:
toy = pd.DataFrame({'Years': [1, 2, 4, 7, 10],
                    'Salary': [1, 2, 3, 8, 9]})     # salary in hundreds of thousands of dollars
toy

**Step 1: list the split points worth trying.** A split "`Years` $< s$" sends every player with fewer than $s$ years to the left box and the rest to the right. Any $s$ between 2 and 4 (say 2.5, 3, or 3.9) puts the same players on each side, so they all score the same. We only need one split point per gap between neighboring values, and the convention is the midpoint. Five distinct values leave four gaps, so there are four split points to try.

**Code worth reading.** `np.unique` sorts the distinct values. `years[:-1]` is every value but the last and `years[1:]` is every value but the first, so averaging them gives the midpoint of each neighboring pair.

In [ ]:
years = np.unique(toy['Years'])
split_points = (years[:-1] + years[1:]) / 2 # clever one-liner for finding midpoints
split_points

**Step 2: score one split.** Take `Years` $< 5.5$, for example (the real algorithm iterates through all possible splits). The first three players go left and the last two go right. Each box predicts its own mean salary, and the score is the total RSS of the two boxes:

| Box | Salaries | Mean | RSS |
|---|---|---|---|
| Left (`Years` $< 5.5$) | 1, 2, 3 | 2 | $(1 - 2)^2 + (2 - 2)^2 + (3 - 2)^2 = 2$ |
| Right (`Years` $\ge 5.5$) | 8, 9 | 8.5 | $(8 - 8.5)^2 + (9 - 8.5)^2 = 0.5$ |
| **Total** | | | $2 + 0.5 = 2.5$ |

**Code worth reading.** The same calculation in pandas. `toy['Years'] < s` is True for the rows in the left box, and `toy.loc[rows, 'Salary']` keeps just their salaries.

In [ ]:
s = 5.5
left = toy.loc[toy['Years'] < s, 'Salary'] # like R_1 in the equations
right = toy.loc[toy['Years'] >= s, 'Salary'] # like R_2 in the equations
left_rss = ((left - left.mean())**2).sum()
right_rss = ((right - right.mean())**2).sum()
print(f'Left box:  salaries {list(left)}, mean {left.mean():.1f}, RSS {left_rss:.1f}')
print(f'Right box: salaries {list(right)}, mean {right.mean():.1f}, RSS {right_rss:.1f}')
print(f'Total RSS: {left_rss + right_rss:.1f}')

**Step 3: score every split and keep the best.** Repeat Step 2 for each of the four split points. The split with the lowest total RSS wins. (With more than one feature, we'd do this for every feature too, and keep the best split overall.)

**Code worth reading.** The loop is Step 2 once per split point. Each pass adds one row to a list of dictionaries, and `pd.DataFrame(rows)` turns that list into a table.

In [ ]:
rows = []
for s in split_points:
    left = toy.loc[toy['Years'] < s, 'Salary']
    right = toy.loc[toy['Years'] >= s, 'Salary']
    rss = ((left - left.mean())**2).sum() + ((right - right.mean())**2).sum()
    rows.append({'split': f'Years < {s}', 'left mean': left.mean(),
                 'right mean': right.mean(), 'total RSS': rss})
pd.DataFrame(rows).round(2)

**Ask:** Which split wins, and does that match what you'd guess from looking at the salaries?

<details>
<summary>Answer</summary>

`Years` $< 5.5$, with a total RSS of 2.5. It matches the eye: the salaries jump from 3 to 8 between 4 and 7 years, so cutting there leaves two boxes whose salaries are each close to their own mean.

</details>

<a id="demo-b"></a>

### Demo B: Finding the best split

Now the real data: 263 players. Pick a feature and slide the split point. The top plot shows each side's mean as a flat line. The bottom plot shows the total RSS for every possible split point on that feature, with the best one starred. The best split over both features is the tree's first split.

In [ ]:
from ch_6_and_8_demos_helpers import best_split_demo

best_split_demo(X_hit, y_hit, size_percent=80)

**Ask:** Compare the best split on `Years` with the best split on `Hits`. Which one does the tree use first? Check against Demo A.

<details>
<summary>Answer</summary>

`Years` $< 4.5$ has the lower RSS (about 115, against about 160 for the best `Hits` split), so it's the first split, exactly as in Demo A's tree.

</details>

**Growing the rest of the tree.** After the first split, the computer repeats the search inside each box separately, and keeps going. This is called **recursive binary splitting**:

```
To grow a tree from a box of training points:
    1. Try every feature and every split point; score each split by the total RSS of its two boxes.
    2. Make the split with the lowest RSS.
    3. Repeat steps 1 and 2 inside each of the two new boxes, until a stopping rule says stop.
```

It's a **greedy** algorithm: it picks the best split *right now* and never revisits it. Checking every *possible* tree would take far too long, so we settle for this greedy approach.

**Ask:** Is the best first split always part of the best possible tree?

<details>
<summary>Answer</summary>

No. Greedy algorithms can miss the best overall answer. A first split that looks only so-so on its own might set up two excellent second splits, and the greedy rule would never find that pair. In practice, greedy trees work well enough, and the ensemble methods (bagging, random forests, and boosting) make up for much of what a single greedy tree misses.

</details>

<a id="section-3"></a>

## 3 · Classification trees: same idea, a different score

For a classification tree, the algorithm is the same, but RSS doesn't make sense for classes (why?). ISL's usual score is the **Gini index** (equation 8.6). For a node $m$, let $\hat p_{mk}$ be the share of the node's training points that are in class $k$. With $K$ classes:

$$
G_m = \sum_{k=1}^{K} \hat p_{mk} (1 - \hat p_{mk})
$$

Each term is small when $\hat p_{mk}$ is near 0 (almost none of class $k$) or near 1 (almost all class $k$), so $G_m$ is small when one class dominates the box. Gini measures **impurity**: how mixed the classes in a box are.

**One way to read it: two draws.** Pull a point out of the box (here, the "box" is a node in the tree we're building) at random, put it back, and pull another. The Gini index is the chance that the two points are different classes. Each term $\hat p_{mk}(1 - \hat p_{mk})$ is "the first point is class $k$" times "the second one isn't." A pure box can never give two different classes, so it scores 0; the more mixed the box, the more often the two draws disagree.

**Two classes.** All of our examples have two classes. Write $\hat p_m$ for the share in class 1; then the share in class 2 is $1 - \hat p_m$, and the two terms of the sum are equal:

$$
G_m = \hat p_m (1 - \hat p_m) + (1 - \hat p_m)\, \hat p_m = 2\, \hat p_m (1 - \hat p_m)
$$

A box with all one class scores 0 (it's **pure**); a 50/50 box scores the most, $2 \times 0.5 \times 0.5 = 0.5$.

A leaf predicts its most common class, and the class shares in the leaf are the tree's probability estimate, just like KNN's vote fraction.

ISL also gives a second measure, **entropy** (equation 8.7), which almost always picks the same splits as Gini. We'll come back to entropy later, when it becomes the training loss for classifiers. In scikit-learn you choose with `criterion='gini'` (the default) or `criterion='entropy'`.

**Ask:** A box holds 8 circles and 2 triangles. What's its Gini index? What would the tree predict there, and with what estimated probability?

<details>
<summary>Answer</summary>

$\hat p = 8/10 = 0.8$, so $G = 2 \times 0.8 \times 0.2 = 0.32$. The tree predicts circle, with an estimated probability of 80%.

</details>

<a id="split-score"></a>

**Scoring a split: a weighted average.** The Gini index scores one box, but a split makes two boxes. To score the split:

 Average the two boxes' Gini indexes, **weighted by how many points each box holds**. If the box being split has $n$ points, and the split sends $n_L$ of them left and $n_R$ right:

$$
\text{split score} = \frac{n_L}{n}\, G_L + \frac{n_R}{n}\, G_R
$$

Here $G_L$ and $G_R$ are the Gini indexes of the left and right boxes, and $\frac{n_L}{n}$ and $\frac{n_R}{n}$ are the shares of the points that went to each. At each level in the tree, we choose the split with the lowest **split score**. 

**Why weight?** RSS didn't need weights: it adds one squared error per point, so a box with more points automatically counts more. The Gini index is computed from *shares*, so it's the same for a box of 2 points as for a box of 200 with the same mixture of class (dependent variable) values. The weights make the box sizes matter. 

Without weighting, **the tree could cheat**. 

Take a box with 50 circles and 50 triangles ($G = 0.5$), and split off a single circle:

| | Left box | Right box |
| --- | --- | --- |
| Points | 1 ○ | 49 ○, 50 △ |
| Gini | 0 | 0.49995 |

- **Unweighted average:** $\frac{1}{2}(0) + \frac{1}{2}(0.49995) \approx 0.25$. This says the split cut the impurity in half.
- **Weighted average:** $\frac{1}{100}(0) + \frac{99}{100}(0.49995) \approx 0.495$. This says almost nothing improved, which is right: 99 of the 100 points are as mixed as before.

**Ask:** A box holds 5 circles and 5 triangles. A split sends 4 circles left, and 1 circle and 5 triangles right. What is the split's score, and how much did it lower the impurity?

<details>
<summary>Answer</summary>

Before the split, the box is 50/50, so $G = 0.5$.

- Left box (4 ○, 0 △): pure, so $G_L = 0$.
- Right box (1 ○, 5 △): $\hat p = 1/6$, so $G_R = 2 \times \frac{1}{6} \times \frac{5}{6} = \frac{10}{36} \approx 0.278$.

Split score: $\frac{4}{10}(0) + \frac{6}{10}(0.278) \approx 0.167$. The impurity dropped from 0.5 to about 0.167.

</details>

<a id="gini-demo"></a>

### Gini demo: moving shapes between boxes

Start with a box of 5 circles and 5 triangles ($G = 0.5$), and split it in two. The sliders choose how many circles and triangles go to the left box; the rest go right.

On the left are the two boxes, each with its Gini index, and the split score. On the right, each box is a dot on the curve $G = 2\hat p(1 - \hat p)$, placed at its share of circles; a bigger dot holds more points. The white square is the box before the split, the orange diamond is the split score, and the arrow between them is the improvement.

Try these:

1. Set the sliders to the split in the Ask above (4 circles and 0 triangles left) and check your answer.
2. Split off a single circle (1 circle, 0 triangles left). The left box is pure, so why is the improvement so small? (This is the "cheat" from [Scoring a split](#split-score), at a smaller size.)
3. Find the split with the biggest improvement.
4. Can you find a split whose score is *higher* than the box before the split?


In [ ]:
from ch_6_and_8_demos_helpers import gini_split_demo

gini_split_demo(circles=5, triangles=5)

<details>
<summary>Answers</summary>

2. The pure box holds only 1 of the 10 points, so it gets a weight of only $\frac{1}{10}$. The other box, with 4 circles and 5 triangles, is still almost 50/50, and it carries $\frac{9}{10}$ of the weight. Split score: $\frac{9}{10} \times 0.494 \approx 0.444$, an improvement of only about 0.056.
3. All 5 circles left and all 5 triangles right (or the reverse). Both boxes are pure, so the split score is 0 and the improvement is the whole 0.5.
4. No. The best you can do is a split that changes nothing, like sending 3 circles and 3 triangles left: both boxes are still 50/50, and the improvement is 0.

**Why the diamond is never above the square.** Check **Show the line between the dots**. The split score is a weighted average of the two dots, so it lies on the straight line between them, closer to the bigger dot. (It sits directly below the square because the box before the split's share of circles is the same weighted average of the two boxes' shares.) The Gini curve bows upward, so a straight line between two points on it never rises above it. That's why a split never makes the Gini split score worse than the box it came from.

</details>


<a id="section-4"></a>

## 4 · Overfitting a Tree 

Back to Chapter 5's simulated data: the same 200 training points (100 circles, 100 triangles), the same 10,000 test points, and the same Bayes boundary.

**Code worth reading.** `DecisionTreeClassifier` with no settings grows until every leaf is pure (or can't be split). When two splits score exactly the same, scikit-learn picks one at random, so `random_state=0` makes the tree repeatable.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from ch_6_and_8_demos_helpers import simulate_data

sim = simulate_data()            # Chapter 5's simulated data
X, y = sim.X, sim.y              # the 200 training points and their classes

full_tree = DecisionTreeClassifier(random_state=0)
full_tree.fit(X, y)
print(f'Leaves: {full_tree.get_n_leaves()}, depth: {full_tree.get_depth()}')
print(f'Training error: {1 - full_tree.score(X, y):.1%}')

<a id="demo-c"></a>

### Demo C: Grow it until it's perfect

`max_depth` caps how many questions the tree may ask on the way down to a leaf. Slide it from 1 to the full tree. Unlike $K$, a bigger value here means a **more** flexible model, so **More flexible** is on the right. Leave both buttons off for now.

The picture works like Demo A's. On the left is the tree: each question, and in each leaf, how many circles (○) and triangles (△) it holds. Past depth 4 the diagram only shows the top four levels. On the right is the same tree as boxes, one per leaf, shaded by the class it predicts. Each step up the slider adds one level of splits, outlined in orange in both panels. (Each step is the full tree cut off at that depth, so moving the slider up only ever adds splits.)

In [ ]:
from ch_6_and_8_demos_helpers import tree_depth_demo

demo_c = tree_depth_demo(sim, depth=1, size_percent=100)

**Notice:** At `max_depth` = 2, the left branch splits on $X_2 \le -0.44$, and *both* of its new boxes still predict circle: one holds 53 circles and 1 triangle, the other 10 circles and 5 triangles. That's allowed. A split doesn't have to change any prediction; it only has to lower the impurity.

**Ask:** Before this split, the box held 63 circles and 6 triangles. The split made the left box purer (53 ○, 1 △) but the right box *less* pure (10 ○, 5 △). How can the split lower the Gini impurity overall? 
<details>
<summary>Answer</summary>

Use the two-class Gini index and the [weighted split score](#split-score) from Section 3.


| Box | Circles | Triangles | Points | $\hat p$ (share of circles) | Gini $= 2\hat p(1 - \hat p)$ |
| --- | --- | --- | --- | --- | --- |
| Before the split | 63 | 6 | 69 | 0.913 | 0.159 |
| Left ($X_2 \le -0.44$) | 53 | 1 | 54 | 0.981 | 0.036 |
| Right ($X_2 > -0.44$) | 10 | 5 | 15 | 0.667 | 0.444 |

The split's score is the weighted average of its two boxes:

$$
\frac{54}{69}(0.036) + \frac{15}{69}(0.444) = 0.028 + 0.097 = 0.125
$$

That's lower than 0.159, so the split lowers the impurity.

**Why it works: the weights.** The box that got purer holds 54 of the 69 points (78%), and it became almost perfectly pure. The box that got worse holds only 15 points (22%). Put another way, the split gathered 5 of the 6 triangles into one small box. The triangles are still outnumbered there, but they're concentrated, and a later split can work on that small box.

**Why the tree doesn't use the error rate.** Count the mistakes. Before the split, the box predicts circle and gets the 6 triangles wrong. After it, both boxes predict circle and get $1 + 5 = 6$ wrong. The error rate didn't change, so a tree that scored splits by error rate would see nothing gained here. The Gini index rewards purer boxes even when no prediction changes, which is why ISL (§8.1.2) recommends it, or entropy, over the error rate for growing a tree.

</details>

**Ask:** The full tree gets every training point right. How will it do on new points from the same recipe?

<details>
<summary>Answer</summary>

Much worse than 0%. The full tree draws small boxes around individual training points, including the ones on the "wrong" side of the Bayes boundary by chance, and new points don't share those quirks.

</details>

Turn on **Show test error** and **Show Bayes boundary**, then slide `max_depth` again. The demo is shown again below.

In [ ]:
demo_c

**Ask:** Which Chapter 5 model did the same thing: 0% training error, and a test error well above the Bayes error rate?

<details>
<summary>Answer</summary>

KNN with $K = 1$ (17.4% test error in Chapter 5; the full tree gets 18.6%). Both memorize the training points.

</details>

A full tree is like $K = 1$: very flexible, and on data like this, high variance. If we drew a new training set, the full tree's boxes would come out very different. As in Chapter 5, that's a statement about the method and its settings (a full tree, 200 points from this recipe), not about one fitted tree.

<a id="section-5"></a>

## 5 · Regularization, kind 1: limits

We want a tree that does well on new data, not one that memorizes. Here's the word for anything we do toward that goal:

> **Regularization** is any change we make to a learning method that is meant to reduce its error on new data, but not its training error.
>
> (Paraphrased from Goodfellow, Bengio, and Courville, *Deep Learning*, 2016, §5.2.2.)

It usually *raises* training error a little. That's the trade: give up some fit to the training data to do better on new data. There are two kinds:

- **Limits** cap how complex the model is allowed to get.
- **Penalties** add a cost for complexity to the training error (Section 6).

scikit-learn's trees have four common limit settings. ISL calls settings like these **tuning parameters**; scikit-learn's documentation, and the broader machine learning field calls them **hyperparameters**.

| Setting | What it caps | More flexible when it's bigger or smaller? |
| --- | --- | --- |
| `max_depth` | the number of questions on the way to any leaf | ? |
| `min_samples_split` | a box needs at least this many points to be split again | ? |
| `min_samples_leaf` | every leaf must hold at least this many training points | ? |
| `max_leaf_nodes` | the total number of leaves | ? |

**Ask:** Fill in the last column. For each setting, does a bigger value or a smaller value let the tree be more flexible? (You can check your answers in Demo D below.)

<details>
<summary>Answers</summary>

| Setting | More flexible when it's | Why |
| --- | --- | --- |
| `max_depth` | bigger | More questions on the way to a leaf, so the boxes can get smaller. |
| `min_samples_split` | smaller | Boxes with only a few points can still be split, so the tree can keep carving. |
| `min_samples_leaf` | smaller | Leaves can be tiny, down to a single point, so the tree can wrap around individual points. |
| `max_leaf_nodes` | bigger | More leaves means more, smaller boxes. |

A pattern to remember: a `max_` setting is an upper limit, so raising it loosens the limit; a `min_` setting is a lower limit, so lowering it loosens the limit. Either way, a looser limit means a more flexible tree.

</details>

KNN's $K$ is the same kind of setting: it caps how closely the model can follow individual points.

<a id="demo-d"></a>

### Demo D: Limits

Each tab is one limit setting. Test error is on now. Watch which way each slider makes the tree more flexible.

In [ ]:
from ch_6_and_8_demos_helpers import limits_demo

limits_demo(sim, size_percent=100)

**Ask:** `min_samples_leaf=40` means every leaf must hold at least 40 training points. Why does that make the tree less flexible?

<details>
<summary>Answer</summary>

A leaf with 40 points can't wrap around one unusual point; its prediction is a vote of at least 40 points. With only 200 points, the tree can have at most 5 such leaves, so it can't carve out the tiny boxes the full tree drew. It's the same idea as a large $K$ in KNN.

</details>

**Code worth reading.** The loop fits one tree per `max_depth` and records both error rates, just like the $K$ loop in Chapter 5. The full tree has depth 9, so the loop stops there.

In [ ]:
X_test, y_test = sim.X_test, sim.y_test     # 10,000 new points from the same recipe

rows = []
for depth in range(1, 10):
    model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    model.fit(X, y)
    rows.append({'max_depth': depth,
                 'training error': 1 - model.score(X, y),
                 'test error': 1 - model.score(X_test, y_test)})
depth_errors = pd.DataFrame(rows).set_index('max_depth')
depth_errors.round(3)

In [ ]:
from ch_6_and_8_demos_helpers import bayes_error_rate, plot_depth_error_curves

plot_depth_error_curves(depth_errors, bayes_error_rate(sim));

It's Chapter 5's picture again, mirrored, because here more flexible is to the right. Training error falls all the way to 0%. Test error falls, then rises. You choose a limit the way you chose $K$: with cross-validation, because with real data there are no 10,000 test points.

<a id="section-6"></a>

## 6 · Regularization, kind 2: a penalty (cost-complexity pruning)

Limits work: in Section 5, a depth limit brought the test error down from 18.6% for the full tree to about 15%. But a limit is a rule about the tree's *shape*, chosen before we know what any split is worth. `max_depth=3` gives every branch exactly three questions, whether that branch needs them or not.

Here are two trees fit to the same 200 training points. Both have 8 leaves, so they're equally complex by that measure. Tree A is the `max_depth=3` tree. Tree B comes from a method we haven't covered yet; this section is about developing this mysterious method.

In [ ]:
from ch_6_and_8_demos_helpers import plot_tree_pair

tree_a = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
tree_b = DecisionTreeClassifier(ccp_alpha=0.011, random_state=0).fit(X, y)   # how we found this one: the rest of this section
plot_tree_pair(sim, {'Tree A: max_depth = 3': tree_a, 'Tree B': tree_b});

**Ask:** Both trees have 8 leaves. Where does each one spend them? Why might Tree B fit the training data better, and do at least as well on new data?

<details>
<summary>Answer</summary>

Tree A has to use exactly three levels everywhere. Every one of its leaves is at depth 3, including at the edges of the plot, where one class already dominates, and in the tangled middle strip, where it has to stop with a leaf of 7 circles and 10 triangles.

Tree B is lopsided. It stops after two questions in the regions that are mostly settled (its leaves with 53 ○ and 1 △, and with 25 ○ and 5 △), and spends the leaves it saved in the middle strip, going five questions deep to separate the circles from the triangles there.

Same number of leaves, placed better: Tree B's training error is 8.5% against Tree A's 10.5%, and its test error is 14.7% against 15.0%. The test gap is small, but the lesson isn't: for the same complexity, *where* the leaves go matters, and the right depth is different in different parts of the feature space.

</details>

**Finding trees like Tree B.** Tree B grows deep where splits pay off and stops where they don't. No single cap on the tree's shape can do that. Instead, we want to judge each split on its merits: *does this split improve the fit enough to be worth one more leaf?*

We already have both halves of that question:

1. **Fit:** the training error of a tree $T$. For a regression tree, that's the RSS from Section 2, added up over all the leaves. For a classification tree, it's the leaves' Gini indexes, weighted by their share of the points.
2. **Size:** $\lvert T \rvert$, the number of leaves of $T$. (The bars mean "number of"; see [Other vertical bars](../../Reference_Material/notation.md#vertical-bars).) Each split adds one leaf.

Neither one alone picks a good tree. Training error alone always prefers more leaves: the full tree wins, with 0%. Size alone always prefers fewer: a single leaf wins. The missing piece is an exchange rate between them: **how much must a split lower the training error to pay for one more leaf?** Call that price $\alpha$.

**Cost-complexity pruning** (ISL equation 8.4) puts the pieces together. For a tree $T$ with leaves (regions) $R_1, \dots, R_{\lvert T \rvert}$:

$$
\text{total cost} = \underbrace{\sum_{m=1}^{\lvert T \rvert} \sum_{i:\, x_i \in R_m} (y_i - \hat y_{R_m})^2}_{\text{fit: training error (RSS)}} \;+\; \underbrace{\alpha \lvert T \rvert}_{\text{size: price per leaf} \,\times\, \text{leaves}}
$$

Here $m$ numbers the leaves, from 1 to $\lvert T \rvert$. The outer sum visits each leaf $R_m$ in turn; the inner sum adds up the squared errors of the training points that land in that leaf.

In words: **total cost = training error + strength × complexity.** Among the candidate trees, keep the one with the lowest total cost. A split survives only if it lowers the training error by more than $\alpha$, the price of the leaf it adds. So a branch can grow as deep as it likes, as long as each split pays its way, and that's how lopsided trees like Tree B come out.

A limit says *you may not* grow past this point. A penalty says *you may, but it costs you*. That's why this is the second kind of regularization: a **penalty**.

ISL writes the formula with RSS, for regression trees. For classification trees, scikit-learn uses the leaves' weighted Gini index as the training error instead. The shape doesn't change.

> **A note on symbols.** ISL calls the pruning strength $\alpha$; scikit-learn calls it `ccp_alpha`. In ridge and lasso regression you'll meet a second penalty that ISL calls $\lambda$ and scikit-learn calls `alpha`. Same idea, different letters.

**How pruning works.** For one value of $\alpha$, the best tree is the subtree of the full tree (the full tree with some branches snipped off) with the lowest total cost. 

But we don't know the best $\alpha$ ahead of time, just as we didn't know the best $K$. So the whole procedure has four steps:

**Optional:**
Understanding these details is totally optional.
```
To prune a tree by cost-complexity pruning:
    1. Grow the full tree, with no limits.
    2. Build the sequence of pruned trees (you don't need to understand the substeps here):
           Start with the full tree (alpha = 0: leaves are free).
           Repeat until only one leaf is left:
               Find the weakest branch: the one that lowers training error
                   the least for each leaf it adds.
               Raise alpha until that branch no longer pays for its leaves.
               Snip it off. Record the new, smaller tree and that alpha.
    3. Choose alpha from the candidate alphas generated in 2. by cross-validation: for each recorded alpha, estimate the test error of pruning at that alpha with k-fold CV.
    4. Refit on all of the training data with the chosen alpha.
```

Notes on the steps:

- **Step 1: why grow everything first?** Why not stop growing as soon as a split isn't worth its price? Because the greedy algorithm can't see ahead: a split that barely helps on its own can set up excellent splits below it. (Remember the $X_2 \le -0.44$ split after [Demo C](#demo-c), which didn't change a single prediction.) Growing everything first lets us judge each branch by everything beneath it.
- **Step 2: why only a few values of $\alpha$?** The best subtree only changes when $\alpha$ passes one of the recorded values. Between two neighboring values, every $\alpha$ gives the same tree. And each tree is the one before it with one more branch snipped off, so they form a single sequence, from the full tree down to one leaf. Snipping the weakest branch each time is why ISL also calls this **weakest-link pruning**.
- **Steps 3 and 4** are the same cross-validation habit as choosing $K$ in Chapter 5. The next few cells do each step in code.

**Code worth reading.** `cost_complexity_pruning_path` does steps 1 and 2: it grows the full tree, then returns the recorded values of $\alpha$ in `ccp_alphas`.

In [ ]:
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X, y)
alphas = path.ccp_alphas
print(f'{len(alphas)} values of alpha, from {alphas[0]:.4f} to {alphas[-1]:.4f}')

<a id="demo-e"></a>

### Demo E: Pruning with a penalty

**Ask (before you move the slider):** If every leaf costs $\alpha$, what happens to the tree as $\alpha$ grows? What happens at a huge $\alpha$?

<details>
<summary>Answer</summary>

The tree loses leaves: the weakest splits stop paying for themselves first. At a huge $\alpha$, no split is worth its price, so the tree is a single leaf that predicts the majority class everywhere.

</details>

The slider steps through the values of $\alpha$ above. The bar on the right is the total cost: the training error measure (blue) plus the penalty, $\alpha \lvert T \rvert$ (orange). Switch the left panel to **Tree** to see the full tree, always drawn in the same layout: each number is the $\alpha$ at which that node becomes a leaf, grey branches are pruned away at the current $\alpha$, and the most recent collapse is outlined in orange.

**Note:** This demo is just steps 1 and 2 about (building the sequence of pruned trees). Choosing the best alpha value comes next. 

In [ ]:
from ch_6_and_8_demos_helpers import pruning_demo

pruning_demo(sim, alphas, size_percent=80)

**Code worth reading.** Choose $\alpha$ by 10-fold cross-validation, with the same `KFold` and `cross_val_score` idiom as Chapter 5 and Lab 4. Then refit on all 200 points with the chosen $\alpha$.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(n_splits=10, shuffle=True, random_state=0)
rows = []
for alpha in alphas[:-1]:        # the last alpha prunes all the way to one leaf
    model = DecisionTreeClassifier(ccp_alpha=alpha, random_state=0)
    scores = cross_val_score(model, X, y, cv=folds)     # accuracy, one score per fold
    rows.append({'alpha': alpha,
                 'leaves': model.fit(X, y).get_n_leaves(),
                 'CV error': 1 - scores.mean()})
pruning = pd.DataFrame(rows)
pruning.round(4)

In [ ]:
best_alpha = pruning.loc[pruning['CV error'].idxmin(), 'alpha']
pruned_tree = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=0).fit(X, y)   # all 200 points
print(f'Chosen alpha: {best_alpha:.4f}, leaves: {pruned_tree.get_n_leaves()}')
print(f'Training error: {1 - pruned_tree.score(X, y):.1%}, '
      f'test error: {1 - pruned_tree.score(X_test, y_test):.1%}')

Cross-validation chose a tree with 8 leaves: Tree B from the start of this section. Its test error is about 4 percentage points lower than the full tree's (14.7% against 18.6%). Several values of $\alpha$ near the bottom have about the same CV error; as with $K$ in Chapter 5, any of them would be fine. What cross-validation reliably does is steer you away from the full tree and from the one-leaf tree.

Limits and penalties get to a similar place by different routes. Both trade a little training error for less variance.

**Ask:** The pruned tree's training error is 8.5%, the full tree's is 0%. Which tree fits the training data better? Which would you use?

<details>
<summary>Answer</summary>

The full tree fits the training data better, but the pruned tree is the one to use: its error on new data is lower. That's regularization doing its job: a little worse on the training data, better on new data.

</details>

**Ask:** Could a limit have found Tree B?

<details>
<summary>Answer</summary>

Here, yes: `max_leaf_nodes=8` gives exactly Tree B. With that setting, scikit-learn grows the tree by always making the most useful split anywhere in the tree, as in Demo A, so it can grow lopsided too.

So why learn the penalty? Two reasons:

1. **It can see ahead.** `max_leaf_nodes` still decides while growing, one greedy step at a time, so it can stop before a weak split that would have set up strong ones. Pruning grows the whole tree first and judges each split by everything beneath it. On this small data set, the two happen to agree.
2. **The shape carries over.** *Training error + strength × complexity* works for models that have no depth or leaves to cap. Ridge and lasso regression use it, and so does weight decay in neural networks.

</details>

<a id="section-7"></a>

## 7 · Two meanings of "regularization"

Many books, including ISL Chapter 6, use *regularization* in a narrower sense: adding a penalty to the training error. That's the kind we'll study next, for linear regression, in ridge and lasso. We'll keep the broad definition too, because it covers what neural networks use: in Weeks 8 and 9, stopping training early and randomly switching off parts of the network are both regularization, and neither is a penalty.

**Ask:** Is choosing $K = 15$ instead of $K = 1$ in KNN regularization, by the broad definition? By the narrow one?

<details>
<summary>Answer</summary>

By the broad definition, yes: it's a limit, chosen to lower error on new data, and it raises training error above 0%. By the narrow definition, no: there's no penalty term added to a training error.

</details>

## 8 · Summary

- A tree predicts by following yes-or-no splits down to a leaf, and it's grown greedily: at each step, make the split that lowers training error (RSS, or the Gini index for classes) the most.
- A tree grown until every leaf is pure has 0% training error and, like $K = 1$, does poorly on new data: on data like this, it's high variance.
- **Regularization** is any change to a method meant to reduce its error on new data, not its training error. Two kinds: **limits** (`max_depth`, `min_samples_leaf`, KNN's $K$) and **penalties**.
- Cost-complexity pruning is a penalty with the shape **training error + strength × complexity**: $\text{RSS} + \alpha \lvert T \rvert$. Choose $\alpha$ by cross-validation. You'll see this shape again in ridge and lasso regression.

# Practice Questions

These questions cover how trees predict and grow, why a full tree overfits, and regularization by limits and by pruning. Questions like these will appear on the learning check, so try each one on paper before you open its hint or answer.

For reference:

- Demo A's tree: if `Years` $\le 4.5$, predict log salary 5.11. Otherwise, if `Hits` $\le 117.5$, predict 6.00; if not, predict 6.74.
- Two-class Gini index of a box: $G = 2\, \hat p (1 - \hat p)$, where $\hat p$ is the share of class 1.
- Score of a split: $\frac{n_L}{n} G_L + \frac{n_R}{n} G_R$, the two boxes' Gini indexes weighted by their share of the points.
- Pruning: total cost = training error + $\alpha \lvert T \rvert$.

### Question 1: Reading a tree

Use Demo A's tree (in the reference list above). (See [Demo A](#demo-a).)

(a) What log salary does the tree predict for a player with 3 years in the league and 200 hits?

(b) What log salary does it predict for a player with 10 years and 90 hits?

<details>
<summary>Hint</summary>

Start at the top question and answer each one in turn. A player stops at the first leaf he reaches.

</details>

<details>
<summary>Answer</summary>

(a) **5.11.** `Years`=3 $\le 4.5$ is true, so he goes left to a leaf right away. His 200 hits never get asked about.

(b) **6.00.** `Years`=10 $\le 4.5$ is false, so go right; `Hits`=90 $\le 117.5$ is true, so go left, to the leaf with 6.00.

</details>

### Question 2: Reading the RSS formula

A split on feature $j$ at split point $s$ is scored by its RSS (ISL equation 8.3):

$$
\text{RSS} = \sum_{i:\, x_i \in R_1(j, s)} (y_i - \hat y_{R_1})^2 \;+\; \sum_{i:\, x_i \in R_2(j, s)} (y_i - \hat y_{R_2})^2
$$

Explain each answer in your own words.

(a) Which part of the formula measures the error in region 1? Which part measures the error in region 2?

(b) 

i. What do the hats in $\hat y_{R_1}$ and $\hat y_{R_2}$ indicate? 

ii. Why does an error formula need them? 

iii. For decision trees, how are they computed?

(c) The tree picks the $j$ and $s$ with the smallest RSS. What does a small RSS say about the two regions?


<details>
<summary>Hint</summary>

Read the subscript under each sum aloud: "the sum over every $i$ such that $x_i$ is in $R_1(j, s)$." (See [A condition under a sum](../../Reference_Material/notation.md#sum-over-a-set).) For the hats, ask what the tree would predict for a player in region 1.

</details>

<details>
<summary>Answer</summary>

$$
\text{RSS} = \underbrace{\sum_{i:\, x_i \in R_1(j, s)} (y_i - \hat y_{R_1})^2}_{\text{Region 1 error}} \;+\; \underbrace{\sum_{i:\, x_i \in R_2(j, s)} (y_i - \hat y_{R_2})^2}_{\text{Region 2 error}}
$$

(a) **The first sum is region 1's error; the second is region 2's.** The subscript under each sum picks out the training points: the first sum adds one term for each training point that lands in region 1 and skips the rest, and the second does the same for region 2. Every training point lands in exactly one of the two regions, so adding the two sums counts each point's error once. The total is the tree's training error after this split.

(b) **A hat marks a prediction.** 

i.  $\hat y_{R_1}$ is what the tree *predicts* for any point in region 1. 

ii. Error measures for predictions always calculate some measure of how much the predictions, $\hat y_{R_m}$ "miss" their corresponding actual values $y_i$.

iii. Decision trees used for regression predict the *mean value* of all the $y_i$'s in the region. So $\hat y_{R_1} = \bar y_i$ for all $y_i \in R_1$. 

Why the mean? Of all the single numbers a region could predict, the mean gives the smallest sum of squared residuals. So each region predicts the best constant it can, and the RSS measures what's left over.

(c) **Each region's responses are tightly clustered around their own mean.** A region's error is small when its training points' responses are all close to one another, since they're then all close to the mean. So a small RSS means the split sorted the points well: similar responses together, low values in one region and high values in the other.

Notice where $j$ and $s$ appear: only in $R_1(j, s)$ and $R_2(j, s)$. Choosing a different feature or split point changes which points land in each region, which changes the two means and the residuals. That's why the tree can score every candidate split with this one formula.

</details>


### Question 3: Scoring one split

Four players have `Years` = 2, 3, 6, 9 and salaries (in hundreds of thousands) 2, 4, 7, 9. Consider the split `Years` $< 4.5$. The left box's mean salary is 3, and the right box's is 8.

(a) What is the RSS of the left box?

(b) What is the total RSS of the split? (See the worked example in [Section 2](#section-2).)

<details>
<summary>Hint</summary>

In each region, add up the squared distances from each salary to that region's mean. Then add the two regions' totals.

</details>

<details>
<summary>Answer</summary>

**Step 1: the equation.** The RSS of a split on feature $j$ at split point $s$ (ISL equation 8.3) is

$$
\text{RSS} = \sum_{i:\, x_i \in R_1(j, s)} (y_i - \hat y_{R_1})^2 \;+\; \sum_{i:\, x_i \in R_2(j, s)} (y_i - \hat y_{R_2})^2
$$

where $\hat y_{R_1}$ is the mean response of the training points in $R_1(j, s)$, and $\hat y_{R_2}$ is the mean in $R_2(j, s)$.

**Step 2: sort the players into regions.** Here $j$ is `Years` and $s = 4.5$, so

$$
R_1 = \{X \mid \texttt{Years} < 4.5\} \quad\text{and}\quad R_2 = \{X \mid \texttt{Years} \ge 4.5\}.
$$

Check each player against the rule:

| `Years` | Salary | `Years` $< 4.5$? | Region |
|---|---|---|---|
| 2 | 2 | yes | $R_1$ |
| 3 | 4 | yes | $R_1$ |
| 6 | 7 | no | $R_2$ |
| 9 | 9 | no | $R_2$ |

So the first sum runs over the two players with salaries 2 and 4, and the second runs over the players with salaries 7 and 9.

**Step 3: each region's mean.**

$$
\hat y_{R_1} = \frac{2 + 4}{2} = 3 \qquad \hat y_{R_2} = \frac{7 + 9}{2} = 8
$$

These are the means the question gave you, and each one is its region's prediction.

**Step 4: add up the squared errors.**

(a) **2.** The left box's RSS is $(2 - 3)^2 + (4 - 3)^2 = (-1)^2 + 1^2 = 1 + 1 = 2$.

(b) **4.** The right box's RSS is $(7 - 8)^2 + (9 - 8)^2 = 1 + 1 = 2$. The split's total RSS is the sum of the two boxes: $2 + 2 = 4$.

</details>

### Question 4: Picking the best split

Here are Question 3's four players plus a fifth, now with `Hits` too. Salaries are in hundreds of thousands.

| Player | `Years` | `Hits` | Salary |
|---|---|---|---|
| A | 2 | 80 | 2 |
| B | 3 | 95 | 4 |
| C | 6 | 150 | 7 |
| D | 9 | 170 | 9 |
| E | 12 | 90 | 3 |

These are all the candidate splits on `Years`, with each split's regions and total RSS already worked out:

| Split | Region 1 salaries | Region 2 salaries | RSS |
|---|---|---|---|
| `Years` $< 2.5$ | 2 | 4, 7, 9, 3 | 22.75 |
| `Years` $< 4.5$ | 2, 4 | 7, 9, 3 | 20.67 |
| `Years` $< 7.5$ | 2, 4, 7 | 9, 3 | 30.67 |
| `Years` $< 10.5$ | 2, 4, 7, 9 | 3 | 29.00 |

(a) If the tree could split only on `Years` (pretend we don't have the `Hits` column), which split would it choose? Why?

(b) Suppose the best split for `Hits` is `Hits` $< 122.5$, with RSS 4.00. Now the tree can split on either feature (we have both columns to work with). Which split does it choose?


<details>
<summary>Hint</summary>

The tree doesn't judge which split looks sensible. It picks whichever split has the smallest RSS.

</details>

<details>
<summary>Answer</summary>

(a) **`Years` $< 4.5$**, because its RSS (20.67) is the smallest of the four. 

(b) **`Hits` $< 122.5$**, because $4.00 < 20.67$. The tree compares the best split on each feature and keeps the one with the smallest RSS. Splitting on `Hits` puts player E in with the other low earners (salaries 2, 4, 3 against 7, 9), so both regions are tight.

That's the whole search in recursive binary splitting: for every feature, try every split point, then keep the single split with the smallest RSS.

</details>


### Question 5: Greedy

True or false: recursive binary splitting tries every possible tree and keeps the one with the lowest training error. Explain in one sentence. (See [Demo B](#demo-b).)

<details>
<summary>Answer</summary>

**False.** It's greedy: at each step it makes the single best split available right now and never revisits it, because checking every possible tree would take far too long.

</details>

### Question 6: Gini index

(a) In your own words, what is the gini index? What is it used for?

(b) A box holds 6 circles and 4 triangles. What's its Gini index?

(c) A second box holds 9 circles and 1 triangle. Is it more or less pure than the first box? (See [Section 3](#section-3).)

<details>
<summary>Hint</summary>

(a) Think about what a Gini index of 0 means, and what plays the role for classification trees that RSS plays for regression trees. (See [Scoring a split](#split-score).)

(b) and (c) With two classes, $G = 2\,\hat p\,(1 - \hat p)$, where $\hat p$ is the share of circles. A pure box scores 0; a 50/50 box scores 0.5.

</details>

<details>
<summary>Answer</summary>

(a) **The Gini index measures impurity: how mixed the classes in a box are.** It's 0 when every point in the box is the same class (pure), and it grows as the classes get more mixed, up to 0.5 for a 50/50 box with two classes.

It's what a classification tree uses to **score candidate splits**, the job RSS does for regression trees. For each candidate split, the tree computes the Gini index of the two new boxes, takes their weighted average (weighted by how many points land in each box), and keeps the split with the smallest score. A good split leaves two boxes that are each mostly one class.

(b) **0.48.** Start from ISL equation 8.6 with two classes, circles and triangles:

$$
G = \hat p_{\text{circle}}(1 - \hat p_{\text{circle}}) + \hat p_{\text{triangle}}(1 - \hat p_{\text{triangle}})
$$

The box has $6 + 4 = 10$ points, so $\hat p_{\text{circle}} = 6/10 = 0.6$ and $\hat p_{\text{triangle}} = 4/10 = 0.4$:

$$
G = 0.6 \times 0.4 + 0.4 \times 0.6 = 0.24 + 0.24 = 0.48
$$

The two terms are equal, which is why the two-class shortcut $G = 2 \times 0.6 \times 0.4 = 0.48$ works.

(c) **More pure.** $\hat p = 9/10 = 0.9$, so $G = 2 \times 0.9 \times 0.1 = 0.18$. That's closer to 0 than 0.48: the second box is mostly circles, while the first is close to an even mix.

</details>


### Question 7: The full tree

In [Demo C](#demo-c), the full tree has 0% training error.

(a) Why can a tree always reach 0% training error (when no two training points share the same features but different classes)?

(b) Its test error was 18.6%, above the Bayes error rate of 10.5%. Which Chapter 5 model behaved the same way, and what is this behavior called?

<details>
<summary>Hint</summary>

What stops a tree from splitting a box again?

</details>

<details>
<summary>Answer</summary>

(a) **It can keep splitting until every leaf is pure.** In the extreme, every training point gets its own box, and each box predicts that point's own class.

(b) **KNN with $K = 1$; overfitting.** Both memorize the training points, including their noise, and on data like this both are high variance.

</details>

### Question 8: Limit, penalty, or neither?

Label each change as a **limit**, a **penalty**, or **not regularization**. (See [Section 5](#section-5) and [Section 6](#section-6).)

(a) Fit the tree with `max_depth=4`.

(b) Fit the tree with `ccp_alpha=0.01`.

(c) Use KNN with $K = 25$ instead of $K = 1$.

(d) Score the same full tree on a larger test set.

<details>
<summary>Hint</summary>

Regularization changes the learning method itself, to lower its error on new data. Does the change cap the model's complexity, or add a cost for it to the training error?

</details>

<details>
<summary>Answer</summary>

(a) **Limit:** it caps the depth.

(b) **Penalty:** every leaf costs $\alpha$.

(c) **Limit:** a larger $K$ caps how closely KNN can follow individual points.

(d) **Not regularization.** The fitted model doesn't change; we only measure it more precisely.

</details>

### Question 9: Which way is more flexible?

For each change, say whether the tree becomes **more** or **less** flexible. (See the table in [Section 5](#section-5).)

(a) Increase `max_depth` from 3 to 6.

(b) Increase `min_samples_leaf` from 5 to 50.

(c) Increase `ccp_alpha` from 0.005 to 0.02.

(d) Increase `max_leaf_nodes` from 4 to 20.

<details>
<summary>Answer</summary>

(a) **More** flexible: more questions per path.

(b) **Less** flexible: bigger leaves can't follow individual points.

(c) **Less** flexible: each leaf costs more, so the pruned tree has fewer leaves.

(d) **More** flexible: more leaves allowed.

</details>

### Question 10: Paying for leaves

Pruning compares two subtrees of the same full tree:

| | Training error measure | Leaves $\lvert T \rvert$ |
| --- | --- | --- |
| Tree A | 0.10 | 10 |
| Tree B | 0.16 | 4 |

(a) With $\alpha = 0.005$, what is each tree's total cost, and which tree does pruning keep?

(b) With $\alpha = 0.02$, which tree does pruning keep? (See [Section 6](#section-6).)

<details>
<summary>Hint</summary>

Total cost = training error + $\alpha \lvert T \rvert$. The lower total wins.

</details>

<details>
<summary>Answer</summary>

(a) Total cost = training error + $\alpha \lvert T \rvert$:

- Tree A: $0.10 + 0.005 \times 10 = 0.10 + 0.05 = 0.15$
- Tree B: $0.16 + 0.005 \times 4 = 0.16 + 0.02 = 0.18$

Tree A has the lower total cost, so **pruning keeps Tree A**. With leaves this cheap, Tree A's 6 extra leaves cost only $6 \times 0.005 = 0.03$, less than the 0.06 of training error they save.

(b) Same formula, with $\alpha = 0.02$:

- Tree A: $0.10 + 0.02 \times 10 = 0.10 + 0.20 = 0.30$
- Tree B: $0.16 + 0.02 \times 4 = 0.16 + 0.08 = 0.24$

Now Tree B has the lower total cost, so **pruning keeps Tree B**. A bigger $\alpha$ makes leaves more expensive: Tree A's 6 extra leaves now cost $6 \times 0.02 = 0.12$, more than the 0.06 of training error they save. So the smaller tree, Tree B, wins.

</details>

### Question 11: Choosing $\alpha$

10-fold cross-validation over the pruning path gives the lowest CV error at $\alpha = 0.0103$. Which tree should you use for predictions on new data: one of the 10 trees fit during cross-validation, or a new tree? (See [Demo E](#demo-e).)

<details>
<summary>Answer</summary>

**A new tree:** `DecisionTreeClassifier(ccp_alpha=0.0103)` fit on all the training data. Cross-validation is for *choosing* $\alpha$; each of its trees saw only 90% of the data.

</details>

### Question 12: Two meanings

ISL Chapter 6 uses "regularization" to mean adding a penalty to the training error. Under that narrow meaning, which of these count: pruning with `ccp_alpha`, and setting `max_depth`? Which count under the broad meaning? (See [Section 7](#section-7).)

<details>
<summary>Answer</summary>

**Narrow: only pruning.** It adds $\alpha \lvert T \rvert$ to the training error. **Broad: both.** Each changes the method to lower its error on new data; `max_depth` is a limit, not a penalty.

</details>